# 04 - Proper Evaluation: Time-Based Splits

Random splits are misleading for time-series data. Let's see why.

In [1]:
import sys
from pathlib import Path

# workshop_3/ holds _fetch.py and _sources.py; src/ holds the project's own helpers.
# When opened in Jupyter Lab with Workshop/ as root, Path.cwd() is Workshop/
# so we need to add workshop_3/ and src/ to the path.
for p in (Path.cwd() / "workshop_3", Path.cwd() / "src"):
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p.resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import _fetch
import _sources as S
from ocean_data_workshop.dsn import dsn
import psycopg

# ML imports
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.model_selection import train_test_split

# Setup
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

print("ML Workshop Environment Ready")
print("=" * 60)
print("Data sources:")
print("  - NDBC 46092 (wind)")
print("  - GLORYS12V1 (ocean)")
print("  - SanctSound MB01 (acoustic)")
print("=" * 60)

ML Workshop Environment Ready
Data sources:
  - NDBC 46092 (wind)
  - GLORYS12V1 (ocean)
  - SanctSound MB01 (acoustic)


## The problem with random splits

Hourly acoustic data is autocorrelated. Adjacent hours are very similar.

A random split puts similar hours in both train and test, making the model look better than it is.

In [2]:
# Load data
import numpy as np

# Load acoustic and dolphin data
conn = psycopg.connect(dsn())

query = "SELECT observed_at, site_id,"
query += " band_25hz, band_50hz, band_160hz, band_500hz, band_2500hz, band_20000hz"
query += " FROM acoustic_tol_hourly WHERE site_id = 'mb01' ORDER BY observed_at"

acoustic_df = pd.read_sql_query(query, conn)

query2 = "SELECT observed_at, presence FROM detection_hourly"
query2 += " WHERE taxon = 'dolphin' AND is_event_list = FALSE ORDER BY observed_at"

dolphin_df = pd.read_sql_query(query2, conn)

conn.close()

# Merge
merged = acoustic_df.merge(dolphin_df, on='observed_at')
X = merged[[c for c in merged.columns if c.startswith('band_')]].values
y = merged['presence'].values

print("Comparing split methods:")
print("=" * 60)

# Time-based split
split_idx = int(len(X) * 0.75)
X_train_t, X_test_t = X[:split_idx], X[split_idx:]
y_train_t, y_test_t = y[:split_idx], y[split_idx:]

# Random split (with stratification)
from sklearn.model_selection import train_test_split
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Train on both
model_t = LogisticRegression(max_iter=2000)
model_t.fit(X_train_t, y_train_t)
pred_t = model_t.predict(X_test_t)
acc_t = accuracy_score(y_test_t, pred_t)

model_r = LogisticRegression(max_iter=2000)
model_r.fit(X_train_r, y_train_r)
pred_r = model_r.predict(X_test_r)
acc_r = accuracy_score(y_test_r, pred_r)

print(f"Time-based split accuracy:  {acc_t:.3f}")
print(f"Random split accuracy:      {acc_r:.3f}")
print(f"Difference:                 {acc_r - acc_t:+.3f}")

# Why does random look better?
print(f"\nTime-based test set range: {merged['observed_at'].iloc[split_idx]} to {merged['observed_at'].iloc[-1]}")
print("Random test set range: random split across entire time range")

# Check if random split has leakage
print("\nRandom split may have leakage because:")
print("  - Adjacent hours are similar (autocorrelation)")
print("  - Model memorizes patterns instead of learning general rules")
print("  - Performance looks better than real-world performance")

Comparing split methods:
Time-based split accuracy:  0.845
Random split accuracy:      0.809
Difference:                 -0.036

Time-based test set range: 2020-02-26 11:00:00+00:00 to 2020-05-31 23:00:00+00:00
Random test set range: random split across entire time range

Random split may have leakage because:
  - Adjacent hours are similar (autocorrelation)
  - Model memorizes patterns instead of learning general rules
  - Performance looks better than real-world performance


/var/folders/6_/6bnkvg2j2qd8720cj2rr98080000gn/T/ipykernel_37059/534699729.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  acoustic_df = pd.read_sql_query(query, conn)
/var/folders/6_/6bnkvg2j2qd8720cj2rr98080000gn/T/ipykernel_37059/534699729.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dolphin_df = pd.read_sql_query(query2, conn)


## Block bootstrapping for confidence intervals

Time-series data needs special handling for confidence intervals.

In [3]:
from sklearn.utils import resample

# Block size (1 hour is too small, use 24 hours for daily patterns)
BLOCK_SIZE = 24

# Load data again for bootstrapping
conn = psycopg.connect(dsn())
query = "SELECT observed_at, site_id,"
query += " band_25hz, band_50hz, band_160hz, band_500hz,"
query += " band_2500hz, band_20000hz,"
query += " band_10000hz, band_12500hz, band_16000hz"
query += " FROM acoustic_tol_hourly WHERE site_id = 'mb01' ORDER BY observed_at"

acoustic_df = pd.read_sql_query(query, conn)

query2 = "SELECT observed_at, presence FROM detection_hourly"
query2 += " WHERE taxon = 'dolphin' AND is_event_list = FALSE ORDER BY observed_at"

dolphin_df = pd.read_sql_query(query2, conn)

conn.close()

merged = acoustic_df.merge(dolphin_df, on='observed_at')
X = merged[[c for c in merged.columns if c.startswith('band_')]].values
y = merged['presence'].values

# Block bootstrap
n_bootstrap = 100
scores = []

for i in range(n_bootstrap):
    # Create blocks
    n_samples = len(X)
    n_blocks = n_samples // BLOCK_SIZE
    
    # Select random blocks
    block_indices = np.random.choice(n_blocks, size=n_blocks, replace=True)
    
    # Reconstruct data from blocks
    X_boot = np.vstack([X[i*BLOCK_SIZE:(i+1)*BLOCK_SIZE] for i in block_indices])
    y_boot = np.hstack([y[i*BLOCK_SIZE:(i+1)*BLOCK_SIZE] for i in block_indices])
    
    # Split and train
    split_idx = int(len(X_boot) * 0.75)
    X_train_b, X_test_b = X_boot[:split_idx], X_boot[split_idx:]
    y_train_b, y_test_b = y_boot[:split_idx], y_boot[split_idx:]
    
    model = LogisticRegression(max_iter=2000)
    model.fit(X_train_b, y_train_b)
    pred_b = model.predict(X_test_b)
    
    scores.append(accuracy_score(y_test_b, pred_b))

# Confidence interval
ci_lower = np.percentile(scores, 2.5)
ci_upper = np.percentile(scores, 97.5)

print(f"Block Bootstrap Results (n={n_bootstrap}):")
print(f"  Mean accuracy:     {np.mean(scores):.3f}")
print(f"  95% CI:            [{ci_lower:.3f}, {ci_upper:.3f}]")
print(f"  Interval width:    {ci_upper - ci_lower:.3f}")

/var/folders/6_/6bnkvg2j2qd8720cj2rr98080000gn/T/ipykernel_37059/45722696.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  acoustic_df = pd.read_sql_query(query, conn)


/var/folders/6_/6bnkvg2j2qd8720cj2rr98080000gn/T/ipykernel_37059/45722696.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dolphin_df = pd.read_sql_query(query2, conn)


Block Bootstrap Results (n=100):
  Mean accuracy:     0.869
  95% CI:            [0.839, 0.893]
  Interval width:    0.054


## The lesson

Time-based splits:
- Realistic evaluation
- No data leakage
- May have smaller test set
- More variance in scores

**Always use time-based splits for time-series data!**